# 03.8 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/03-classification/08-multiclass/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

## Exercise 1

$e^3 = 20.09$, $e^3 = 20.09$, $e^0 = 1$; sum $= 41.17$. Softmax $= (0.488, 0.488, 0.024)$. Tied scores get identical probabilities, and a score gap of 3 becomes a factor of $e^3 \approx 20$, so the third class is nearly ruled out.

In [2]:
z = np.array([3.0, 3.0, 0.0]); print((np.exp(z) / np.exp(z).sum()).round(3))

[0.488 0.488 0.024]


## Exercise 2

New table: standard row becomes (1, 3, 0). Diagonal = 2 + 3 + 2 = 7 → **accuracy 7/9 = 0.778**.

- premium: precision 2/3, recall 2/3, F1 0.667
- standard: precision 3/4 = 0.75, recall 3/4 = 0.75, **F1 0.75**
- none: precision 2/2 = 1.0, recall 2/2 = 1.0, F1 1.0

Macro F1 = (0.667 + 0.75 + 1.0)/3 = **0.806**; weighted = (3 × 0.667 + 4 × 0.75 + 2 × 1.0)/9 = **0.778**. One corrected prediction fixed a standard error *and* removed none's only false positive, so two classes improved.

In [3]:
from sklearn.metrics import classification_report
labels = ["premium", "standard", "premium", "standard", "standard", "standard", "none", "none", "premium"]
preds  = ["premium", "standard", "standard", "premium", "standard", "standard", "none", "none", "premium"]
print(classification_report(labels, preds, labels=["premium", "standard", "none"], digits=3))

              precision    recall  f1-score   support

     premium      0.667     0.667     0.667         3
    standard      0.750     0.750     0.750         4
        none      1.000     1.000     1.000         2

    accuracy                          0.778         9
   macro avg      0.806     0.806     0.806         9
weighted avg      0.778     0.778     0.778         9



## Exercise 3

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
iris = load_iris(as_frame=True)
Xi, yi = iris.data, iris.target_names[iris.target]
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
for k in [1, 5, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(Xi_train, yi_train)
    print(f"k={k:>2}: macro F1 {f1_score(yi_test, m.predict(Xi_test), average='macro'):.3f}")
best = KNeighborsClassifier(n_neighbors=5).fit(Xi_train, yi_train)
print(pd.DataFrame(confusion_matrix(yi_test, best.predict(Xi_test), labels=best.classes_), index=best.classes_, columns=best.classes_))

k= 1: macro F1 0.978
k= 5: macro F1 1.000
k=15: macro F1 0.978
            setosa  versicolor  virginica
setosa          15           0          0
versicolor       0          15          0
virginica        0           0         15


All three k values score near-perfectly (k = 5 gets every test flower right on this split); the single error at k = 1 and k = 15 is a versicolor ↔ virginica mix-up. Setosa is linearly separable from both.

## Exercise 4

In [5]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd, numpy as np
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
from sklearn.model_selection import train_test_split

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")          # fast cache written by lesson 01.3
raw = pd.read_csv(csv, parse_dates=["InvoiceDate"]) if csv.exists() else pd.read_excel(xlsx, sheet_name="Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%, train 2,487 / test 830


In [6]:
# three-way target from next-quarter spend: none (£0), standard (£0-500), premium (>£500)
cust["catalogue"] = pd.cut(future_spend.reindex(cust.index).fillna(0), bins=[-1, 0, 500, np.inf],
                           labels=["none", "standard", "premium"])
print(cust["catalogue"].value_counts())
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cust[features], cust["catalogue"], test_size=0.25, random_state=42, stratify=cust["catalogue"])

catalogue
none        1446
premium     1060
standard     811
Name: count, dtype: int64


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, f1_score
order = ["none", "standard", "premium"]
for cw in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight=cw)).fit(Xc_train, yc_train)
    p = m.predict(Xc_test)
    rec = recall_score(yc_test, p, labels=order, average=None)
    print(f"class_weight={str(cw):<9} recall none/standard/premium = {rec.round(3)}   macro F1 {f1_score(yc_test, p, average='macro'):.3f}   accuracy {(p == yc_test).mean():.3f}")

class_weight=None      recall none/standard/premium = [0.859 0.158 0.57 ]   macro F1 0.517   accuracy 0.595
class_weight=balanced  recall none/standard/premium = [0.635 0.468 0.528]   macro F1 0.549   accuracy 0.560


`balanced` gives rarer classes more weight during training. Recall for the smallest class, *standard* (811 customers), roughly triples, at the expense of recall on *none* (the largest class) and of overall accuracy; macro F1 goes up, accuracy goes down. Whether that trade is worth it depends on the cost of each mistake, as in 03.7.